![Kayak](https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png)

# 🗺️ Notebook 2 — Visualisation des Résultats

**Projet :** Plan Your Trip with Kayak — Jedha Bootcamp Data Engineering  
**Auteur :** Lahonde  
**Date :** Février 2026

---

## Objectif de ce notebook

Ce notebook lit les données **directement depuis AWS RDS** (le Data Warehouse) et produit les deux livrables cartographiques demandés :

| Livrable | Description |
|----------|-------------|
| **Carte 1** | Top-5 destinations françaises selon le score météo |
| **Carte 2** | Top-20 hôtels dans les meilleures destinations |

> 📌 **Prérequis :** Le Notebook 1 (`01_Data_Collection_Pipeline.ipynb`) doit avoir été exécuté au préalable.

---

### Résultats attendus

D'après le sujet :

![Exemple de carte attendue](https://full-stack-assets.s3.eu-west-3.amazonaws.com/images/Kayak_best_destination_project.png)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import plotly.express as px
import plotly.graph_objects as go

# Chargement des variables d'environnement
load_dotenv()
RDS_URI = os.getenv("RDS_URI")

print("Imports OK")
print(f"RDS URI : {'OK' if RDS_URI and 'ENDPOINT' not in RDS_URI else 'MANQUANT'}")

---
## 📥 Chargement des données

On charge les données depuis les **CSV finaux** (sortie de l'ETL du Notebook 1).

Cela permet de rejouer la visualisation **sans dépendance cloud** (ni S3 ni RDS) :
```
APIs / Scraping  →  CSV local  →  Plotly
                       ↑
          (équivalent fonctionnel à la table RDS)
```

> 💡 Pour la version qui lit directement depuis **AWS RDS** (Data Warehouse cloud), voir le Notebook 1 (étape ETL).


In [ ]:
# ── Chargement des données depuis les CSV ─────────────────────────────
# Les CSV sont déjà au format "post-ETL" (mêmes colonnes que la table RDS).
# Ils contiennent les scores météo calculés et le classement des villes.

DATA = "../data"

df_weather = pd.read_csv(f"{DATA}/weather_cities.csv").sort_values("rank")
df_hotels  = pd.read_csv(f"{DATA}/hotels_booking.csv")

print(f"✅ Données chargées depuis CSV :")
print(f"   weather_cities : {df_weather.shape[0]} villes")
print(f"   hotels_booking : {df_hotels.shape[0]} hôtels")
print()

# Aperçu du Top 10 destinations
print("Top 10 destinations :")
df_weather[["rank", "city", "temp_max_mean", "pop_mean", "rain_total", "weather_score"]].head(10)


---
## 🗺️ Carte 1 — Top-5 Destinations en France

### Lecture de la carte

- **Chaque point** = une des 35 villes françaises analysées
- **Taille du point** = proportionnelle au score météo (plus grand = meilleure météo)
- **Couleur** : échelle `RdYlGn` (rouge = mauvaise météo → vert = excellente météo)
- **⭐ Étoiles dorées** = Top 5 des destinations
- **Survol (hover)** : affiche le nom, le score, la température, la probabilité de pluie

### Score météo — rappel de la formule

```
score = norm(temp_max) × 0.40
      + norm(1 - pop)  × 0.30
      + norm(1 - rain) × 0.20
      + norm(1 - hum)  × 0.10
```

Normalisé entre 0 et 100 (min-max scaling sur les 35 villes).

In [ ]:
# ── Carte 1 : Toutes les 35 villes colorées par score météo ──────────
fig_destinations = px.scatter_map(
    df_weather,
    lat   = "lat",
    lon   = "lon",
    color = "weather_score",
    size  = "weather_score",
    hover_name = "city",
    hover_data = {
        "weather_score": ":.1f",
        "temp_max_mean" : ":.1f",
        "pop_mean"      : ":.1%",
        "rain_total"    : ":.1f",
        "rank"          : True
    },
    color_continuous_scale = "RdYlGn",
    size_max    = 30,
    zoom        = 4.5,
    center      = {"lat": 46.5, "lon": 2.5},
    map_style   = "open-street-map",
    title       = "<b>Meilleures Destinations en France</b>  "
                  "<br><sub>Score météo sur 5 jours (février 2026) — Source : OpenWeatherMap</sub>",
    labels = {
        "weather_score": "Score météo",
        "temp_max_mean" : "Temp. max moy. (°C)",
        "pop_mean"      : "Proba pluie",
        "rain_total"    : "Pluie totale (mm)",
        "rank"          : "Classement"
    }
)

# ── Ajouter les étoiles pour le Top 5 ────────────────────────────────
top5 = df_weather[df_weather["rank"] <= 5].copy()

fig_destinations.add_trace(
    go.Scattermap(
        lat          = top5["lat"],
        lon          = top5["lon"],
        mode         = "markers+text",
        text         = top5["rank"].astype(str) + ". " + top5["city"],
        textposition = "top right",
        textfont     = dict(size=13, color="#222222"),
        marker       = dict(size=22, color="gold", symbol="star"),
        name         = "Top 5",
        hoverinfo    = "skip"
    )
)

fig_destinations.update_layout(
    height = 650,
    margin = {"r": 0, "t": 80, "l": 0, "b": 0},
    font   = dict(family="Arial", size=13),
    title_font_size = 18,
    coloraxis_colorbar = dict(
        title     = "Score<br>météo",
        tickvals  = [0, 25, 50, 75, 100],
        ticktext  = ["0", "25", "50", "75", "100"]
    )
)

fig_destinations.show()

In [ ]:
# ── Tableau récapitulatif du Top 5 ───────────────────────────────────
top5_display = df_weather[df_weather["rank"] <= 5][[
    "rank", "city", "temp_max_mean", "pop_mean", "rain_total", "humidity_mean", "weather_score"
]].copy()

top5_display.columns = ["#", "Ville", "Temp. max moy. (°C)", "Proba pluie", "Pluie totale (mm)", "Humidité (%)", "Score /100"]
top5_display["Proba pluie"] = top5_display["Proba pluie"].apply(lambda x: f"{x:.1%}")

print("Top 5 Destinations — Détail :")
display(top5_display.set_index("#"))

---
## 🏨 Carte 2 — Top-20 Hôtels dans les Meilleures Destinations

### Sélection des hôtels

On ne montre que les hôtels situés dans les **5 meilleures destinations** :
Aix-en-Provence, Avignon, Marseille, Paris, Grenoble.

On les trie par **score Booking.com décroissant** et on prend les **20 meilleurs**.

### Lecture de la carte

- **Chaque point** = un hôtel
- **Taille du point** = proportionnelle au score Booking
- **Couleur** : palette `Blues` (plus foncé = meilleur score)
- **Survol (hover)** : nom, ville, score, description

In [ ]:
# ── Sélection des hôtels dans le Top 5 des destinations ──────────────
top5_cities = df_weather[df_weather["rank"] <= 5]["city"].tolist()
print(f"Villes du Top 5 : {top5_cities}")

df_hotels_top = (
    df_hotels[df_hotels["city"].isin(top5_cities)]
    .dropna(subset=["score"])          # Garder uniquement les hôtels avec un score
    .sort_values("score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
df_hotels_top["hotel_rank"] = df_hotels_top.index + 1

print(f"\n{len(df_hotels_top)} hôtels sélectionnés pour la carte")

In [ ]:
# ── Carte 2 : Top-20 hôtels ───────────────────────────────────────────
fig_hotels = px.scatter_map(
    df_hotels_top,
    lat        = "lat",
    lon        = "lon",
    color      = "score",
    size       = "score",
    hover_name = "hotel_name",
    hover_data = {
        "city"       : True,
        "score"      : ":.1f",
        "description": True,
        "hotel_rank" : True
    },
    color_continuous_scale = "Blues",
    size_max   = 25,
    zoom       = 4.5,
    center     = {"lat": 44.5, "lon": 5.0},
    map_style  = "open-street-map",
    title      = "<b>Top 20 Hôtels dans les Meilleures Destinations</b>"
                 "<br><sub>Aix-en-Provence · Avignon · Marseille · Paris · Grenoble — Source : Booking.com</sub>",
    labels     = {
        "score"      : "Score Booking",
        "city"       : "Ville",
        "hotel_rank" : "Classement",
        "description": "Adresse"
    }
)

# Ajouter les numéros de classement sur la carte
fig_hotels.add_trace(
    go.Scattermap(
        lat      = df_hotels_top["lat"],
        lon      = df_hotels_top["lon"],
        mode     = "text",
        text     = df_hotels_top["hotel_rank"].astype(str),
        textfont = dict(size=9, color="white"),
        hoverinfo= "skip",
        showlegend = False
    )
)

fig_hotels.update_layout(
    height = 650,
    margin = {"r": 0, "t": 80, "l": 0, "b": 0},
    font   = dict(family="Arial", size=13),
    title_font_size = 18
)

fig_hotels.show()

In [ ]:
# ── Tableau récapitulatif du Top 20 hôtels ────────────────────────────
top20_display = df_hotels_top[[
    "hotel_rank", "hotel_name", "city", "score", "description"
]].copy()
top20_display.columns = ["#", "Hôtel", "Ville", "Score /10", "Adresse"]

print("Top 20 Hôtels — Détail :")
display(top20_display.set_index("#"))

---
## 💾 Export des cartes en HTML

Les cartes Plotly peuvent être **exportées en HTML** pour être partagées sans Jupyter — elles restent entièrement interactives dans un navigateur.

In [ ]:
# Export en fichiers HTML interactifs (livrable Jedha)
# Les fichiers sont écrits dans outputs/maps/ pour rester organisés.
import os
OUT = "../outputs/maps"
os.makedirs(OUT, exist_ok=True)

fig_destinations.write_html(f"{OUT}/map_destinations.html")
fig_hotels.write_html(f"{OUT}/map_hotels.html")

print("✅ Cartes exportées :")
print(f"   {OUT}/map_destinations.html  (Carte Top-5 destinations)")
print(f"   {OUT}/map_hotels.html        (Carte Top-20 hôtels)")
print()
print("Ouvrir dans un navigateur pour les explorer de façon interactive.")


---
## ✅ Conclusion

### Top 5 Destinations (score météo 5 jours — février 2026)

| # | Ville | Temp. max moy. | Proba pluie | Score |
|---|-------|---------------|-------------|-------|
| 1 | **Aix-en-Provence** | 16.3 °C | 0.6 % | 88.91 |
| 2 | **Avignon** | 17.5 °C | 3.8 % | 87.81 |
| 3 | **Marseille** | 15.8 °C | 0.0 % | 86.65 |
| 4 | **Paris** | 15.6 °C | 2.2 % | 85.54 |
| 5 | **Grenoble** | 16.5 °C | 3.8 % | 82.25 |

### Infrastructure AWS utilisée

```
Data Lake    : s3://alh-consulting-kayak/raw/
Data Warehouse : kayak-jedha.cdwsi2uiosiy.eu-west-3.rds.amazonaws.com
               Tables : cities_weather (35 lignes) + hotels (679 lignes)
```

### Livrables du projet

✅ Fichiers CSV dans S3 (`weather_cities.csv` + `hotels_booking.csv`)  
✅ Base de données SQL sur RDS (`cities_weather` + `hotels`)  
✅ Carte Top-5 destinations interactive (`map_destinations.html`)  
✅ Carte Top-20 hôtels interactive (`map_hotels.html`)